<img src='https://raw.githubusercontent.com/ACTRIS-Data-Centre/actris-jupyter-hub/refs/heads/main/actris-dc-logo.png' width=500 align=right>

# Access and use ACTRIS In-Situ Data for Wildfires

Search and plot fully quality-controlled ACTRIS aerosol in-situ measurements: absorption,
scattering, particle number and size distribution, and chemical composition.

Pick an instrument, a variable, a facility and an instrument model from the menus; the matching
datasets are found, loaded and plotted below. The map shows every facility that has data for the
current instrument and variable, with the selected one highlighted.

All the logic lives in the helper module `ACTRIS_insitu.py` next to this notebook; the notebook
only drives it. *Module and plotting by Lise Eder Murberg and Maria Leseth Føyen (NILU).*

> **Clearing the outputs.** Use *Edit → Clear Outputs of All Cells* in two situations:
>
> * **before running the notebook again**, so that the interactive widgets are rebuilt from scratch.
>   Widget state belongs to the kernel that created it, so leftovers from a previous run can make
>   controls stop responding or disappear;
> * **before saving**, so that the file stays small and does not carry stale widget state. Plotly
>   embeds a large JavaScript bundle into saved outputs, and reopening a notebook that stores widget
>   state shows "Error displaying widget: model not found".
>
> If controls are still unresponsive after clearing, restart the kernel (*Kernel → Restart Kernel*)
> and, should that not be enough, reload the browser page: the widgets live partly in the page
> itself, and a reload is sometimes the only way to clear what is left there.

## 0. Configuration

* `INSTALL_MISSING` — install the Python packages that are missing, and only those
* `DEV_RELOAD` — reload the helper module on every run; only needed while editing
  `ACTRIS_insitu.py`, and it re-injects the Plotly bundle into the outputs, so keep it `False`

In [ ]:
# The kernel keeps a handle on the directory it was started in. If that directory is replaced
# while the kernel is running - the shared workspace being republished, for instance - every
# relative path fails with "Stale file handle" or FileNotFoundError. Recover instead of leaving
# the user with an error that has nothing to do with what they were doing.
import os

try:
    os.getcwd()
except OSError:
    _nb = os.environ.get("JPY_SESSION_NAME", "")
    _target = os.path.dirname(_nb) if _nb else os.path.expanduser("~")
    if not os.path.isdir(_target):
        _target = os.path.expanduser("~")
    os.chdir(_target)
    print(f"The working directory had become stale and was reset to {_target}.")
    print("If anything still looks wrong, restart the kernel (Kernel -> Restart Kernel).")

# Identity of this notebook. The file itself carries no version number - only one version is
# published at a time - so the version is printed here instead, which is also what tells you
# whether the kernel is running the file you think it is.
NOTEBOOK_NAME = "ACTRIS Aerosol In-Situ Level 2"
NOTEBOOK_VERSION = "3.2"
NOTEBOOK_DATE = "2026-11-22"

INSTALL_MISSING = True
DEV_RELOAD = False

print(f"{NOTEBOOK_NAME} - version {NOTEBOOK_VERSION} ({NOTEBOOK_DATE})")

## 1. Dependencies and helper module

Only the packages that are actually missing are installed. Each VRE user has their own container,
so a notebook that runs for a colleague can still fail here with `ModuleNotFoundError`.

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = {
    "numpy": "numpy",
    "requests": "requests",
    "xarray": "xarray",
    "plotly": "plotly",
    "tqdm": "tqdm",
    "ipywidgets": "ipywidgets",
    "ebas_plot": "ebas-plot",      # plotting package maintained by NILU
    # Hidden dependencies, not imported by name anywhere:
    #   netCDF4  - backend xarray uses to read the OPeNDAP datasets
    #   nbformat - plotly needs it to render figures, but does not declare it
    "netCDF4": "netCDF4",
    "nbformat": "nbformat",
}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]

if not missing:
    print("--> All required packages are already installed.")
elif not INSTALL_MISSING:
    print(f"--> Missing packages: {', '.join(missing)} (INSTALL_MISSING is False)")
else:
    print(f"--> Installing missing packages: {', '.join(missing)}")
    # pip reads os.getcwd() at startup; running it from the home directory keeps it working
    # even if the folder the kernel started in has since been replaced.
    result = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing],
                            capture_output=True, text=True,
                            cwd=os.path.expanduser("~"))
    print("--> Done." if result.returncode == 0 else result.stderr[-2000:])

# ebas-plot is updated regularly; run this once in a while to pick up a new release:
#     %pip install -q --upgrade ebas-plot

In [ ]:
import importlib
from pathlib import Path

if not Path("ACTRIS_insitu.py").is_file():
    raise FileNotFoundError(
        "ACTRIS_insitu.py not found in the working directory. This notebook must sit next to "
        "the helper module."
    )

import ACTRIS_insitu as insitu

if DEV_RELOAD:
    importlib.reload(insitu)
    print("Helper module reloaded (DEV_RELOAD is True).")

## 2. Load the metadata

The facility and dataset metadata are read from the ACTRIS metadata API
(`prod-actris-md.nilu.no`) for every instrument/variable combination this notebook covers. It takes
a few seconds and is done once per session — there is no local cache to keep in sync any more.

In [ ]:
insitu.load_metadata()

## 3. Select, load and plot

The menus open on a worked example — absorption coefficient from the filter absorption photometer
at Sonnblick during the 2025 Canadian fire season (May to September 2025) — so that pressing
**Confirm Selection** straight away gives a meaningful plot. Every
choice is yours to change, and a default is ignored when the metadata do not contain it. They are
set in `DEFAULT_SELECTION` at the top of `ACTRIS_insitu.py`.

Each choice narrows the following one, and the map shows the facilities that have data for the
current instrument and variable.

**Period.** Only the datasets that cover the chosen period are opened, and only that period is
plotted. Leave both dates empty to see the whole record. A facility can have more than ten years of
hourly data: drawn at full resolution that is millions of values, more than a browser page can
hold, so a figure that would be too large is shown as daily (or weekly, or monthly) means instead,
and a line above it says so. A shorter period shows every measurement.

Pressing **Confirm Selection** finds the matching datasets, loads and concatenates them, and plots
them directly below the menus. Loading several years over OPeNDAP can take a while; the button
shows *Loading...* until the figure arrives.

If the widgets ever fail to render — typically after a kernel restart, or when reopening a notebook
whose outputs were saved — clear the outputs and re-run this cell.

In [ ]:
instrument_dd, variable_dd, station_dd, model_dd, btn = insitu.create_search_widgets()

<a id="export"></a>
## Keeping the results

This notebook **keeps nothing between runs**: the data are read from the server as they are
plotted, and the figures are interactive Plotly charts that live only in this page. There is no
output folder to empty and no export cell.

**To keep a figure**, hover over it and use the **camera icon** in the toolbar that appears in the
top-right corner: it saves a PNG to your browser's download folder. The same toolbar offers zoom,
pan and autoscale, worth using on a long record.

**To keep the data instead**, run the cell below after a selection has been plotted. It writes the
data of the chosen period, at full resolution even when the figure shows averages, as a NetCDF
file next to this notebook, which you can then download from the
file browser on the left (right-click the file, *Download*).

In [ ]:
import shutil
from pathlib import Path

# The module writes the data of the chosen period to /tmp, at full resolution, before plotting
# them; copying that file is immediate and avoids downloading everything a second time.
tmp_nc = Path("/tmp/concatenated_datasets.nc")
nc_path = Path("insitu_selection.nc")

if tmp_nc.exists():
    shutil.copy2(tmp_nc, nc_path)
    print(f"Saved to {nc_path.resolve()} ({nc_path.stat().st_size / 1e6:.1f} MB)")
    print("Download it from the file browser on the left: right-click the file, then Download.")
else:
    print("Nothing to save yet: make a selection above and press Confirm Selection first.")